In [21]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [22]:
df_train = pd.read_csv(r"D:\Digilians\competition\train.csv")
df_test = pd.read_csv(r"D:\Digilians\competition\test.csv")

In [23]:
submission = df_test[['id']]

In [24]:
df_train.drop('id', axis=1, inplace=True)
df_test.drop('id', axis=1, inplace=True)

In [25]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_train['Irrigation_Need'] = df_train['Irrigation_Need'].map(target_map)

In [26]:
y_train = df_train['Irrigation_Need']
df_train.drop('Irrigation_Need', axis=1, inplace=True)

In [27]:
# Feature Engineering
df_train['Temp_Humidity'] = df_train['Temperature_C'] * df_train['Humidity']
df_test['Temp_Humidity'] = df_test['Temperature_C'] * df_test['Humidity']

df_train['Rain_Soil'] = df_train['Rainfall_mm'] / (df_train['Soil_Moisture'] + 1)
df_test['Rain_Soil'] = df_test['Rainfall_mm'] / (df_test['Soil_Moisture'] + 1)

df_train['Evap_Index'] = (df_train['Temperature_C'] * df_train['Wind_Speed_kmh'] * df_train['Sunlight_Hours']) / (df_train['Humidity'] + 1)
df_test['Evap_Index'] = (df_test['Temperature_C'] * df_test['Wind_Speed_kmh'] * df_test['Sunlight_Hours']) / (df_test['Humidity'] + 1)

df_train['Soil_Thirst'] = (df_train['Temperature_C']**2) / (df_train['Soil_Moisture'] + df_train['Previous_Irrigation_mm'] + 1)
df_test['Soil_Thirst'] = (df_test['Temperature_C']**2) / (df_test['Soil_Moisture'] + df_test['Previous_Irrigation_mm'] + 1)

In [28]:
lst=['Soil_pH','Sunlight_Hours','Organic_Carbon',
     'Previous_Irrigation_mm','Field_Area_hectare','Humidity','Electrical_Conductivity']

df_train.drop(lst, axis=1, inplace=True)
df_test.drop(lst, axis=1, inplace=True)

In [29]:
# Categorical Encoding
cat_cols = df_train.select_dtypes(include='object').columns

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

df_train[cat_cols] = oe.fit_transform(df_train[cat_cols])
df_test[cat_cols] = oe.transform(df_test[cat_cols])

In [30]:
# Scaling
scaler = StandardScaler()

df_train = pd.DataFrame(scaler.fit_transform(df_train), columns=df_train.columns)
df_test = pd.DataFrame(scaler.transform(df_test), columns=df_test.columns)


In [ ]:
# SMOTE
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(df_train, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_resampled))


In [ ]:
# Class Weights
classes = np.unique(y_resampled)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_resampled
)

class_weights = dict(zip(classes, weights))

print(class_weights)

In [ ]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_resampled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Training
history = model.fit(
    X_resampled, y_resampled,
    validation_split=0.1,
    epochs=50,
    batch_size=1024,
    callbacks=[early_stopping, lr_scheduler],
    class_weight=class_weights
)

In [ ]:
# Prediction
predictions = model.predict(df_test)
predicted_classes = np.argmax(predictions, axis=1)

In [36]:
reverse_map = {0: "Low", 1: "Medium", 2: "High"}

submission['Irrigation_Need'] = predicted_classes
submission['Irrigation_Need'] = submission['Irrigation_Need'].map(reverse_map)

submission.to_csv(r"D:\Digilians\competition\submission.csv", index=False)